In [1]:
!pip install pandas numpy matplotlib scikit-learn plotly

In [3]:
from google.colab import files
uploaded = files.upload()

Saving PublicationofBuildingEnergyPerformanceData.zip to PublicationofBuildingEnergyPerformanceData.zip


In [4]:
import zipfile

zip_path = "PublicationofBuildingEnergyPerformanceData.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("energy_data")

print("Files extracted successfully")

Files extracted successfully


In [5]:
import pandas as pd

df = pd.read_csv("energy_data/Listing of Building Energy Performance Data for Commercial Buildings.csv")

df.head()

,buildingname,buildingaddress,buildingtype,greenmarkstatus,greenmarkrating,greenmarkyearaward,buildingsize,grossfloorarea,2017energyuseintensity,2018energyusintensity,voluntarydisclosure
0,HEXACUBE,"160 CHANGI ROAD, SINGAPORE 419728",Mixed Development,No,NaN,NaN,Small,"5,036",81.0,105.0,Y
1,NaN,NaN,Retail,No,NaN,NaN,Small,NaN,475.0,402.0,N
2,CITY SQUARE MALL,"180 KITCHENER ROAD, SINGAPORE 208539",Retail,Yes,Platinum,2018.0,Large,"65,640",382.0,365.0,Y
3,REPUBLIC PLAZA,"9 RAFFLES PLACE, SINGAPORE 048619",Office,Yes,Platinum,2018.0,Large,"102,356",212.0,183.0,Y
4,CENTRAL MALL,"1 MAGAZINE ROAD, SINGAPORE 059567",Office,Yes,Platinum,2017.0,Large,"15,769",203.0,181.0,Y


In [7]:
df = df.rename(columns={
    "2017energyuseintensity": "energy_intensity"
})

df = df.dropna(subset=["energy_intensity"])

df.head()

,buildingname,buildingaddress,buildingtype,greenmarkstatus,greenmarkrating,greenmarkyearaward,buildingsize,grossfloorarea,energy_intensity,2018energyusintensity,voluntarydisclosure
0,HEXACUBE,"160 CHANGI ROAD, SINGAPORE 419728",Mixed Development,No,NaN,NaN,Small,"5,036",81.0,105.0,Y
1,NaN,NaN,Retail,No,NaN,NaN,Small,NaN,475.0,402.0,N
2,CITY SQUARE MALL,"180 KITCHENER ROAD, SINGAPORE 208539",Retail,Yes,Platinum,2018.0,Large,"65,640",382.0,365.0,Y
3,REPUBLIC PLAZA,"9 RAFFLES PLACE, SINGAPORE 048619",Office,Yes,Platinum,2018.0,Large,"102,356",212.0,183.0,Y
4,CENTRAL MALL,"1 MAGAZINE ROAD, SINGAPORE 059567",Office,Yes,Platinum,2017.0,Large,"15,769",203.0,181.0,Y


In [9]:
campus_energy = df.groupby("buildingtype")["energy_intensity"].mean().reset_index()

campus_energy.head()

,buildingtype,energy_intensity
0,Community Hospital,210.250000
1,General Hospital/ Specialist Centre (Public),366.111111
2,Hotel,274.214286
3,ITE,106.666667
4,Mixed Development,308.700000


In [10]:
import numpy as np
from sklearn.linear_model import LinearRegression

campus_energy["index"] = np.arange(len(campus_energy))

X = campus_energy[["index"]]
y = campus_energy["energy_intensity"]

model = LinearRegression()
model.fit(X, y)

campus_energy["regression_pred"] = model.predict(X)

In [11]:
campus_energy["smooth_pred"] = campus_energy["energy_intensity"].rolling(3).mean()

In [12]:
campus_energy["ensemble_pred"] = (
    campus_energy["regression_pred"]*0.5 +
    campus_energy["smooth_pred"]*0.5
)

In [14]:
import plotly.express as px

fig = px.bar(
    campus_energy,
    x="buildingtype",
    y="energy_intensity",
    title="Campus Energy Usage by Building Type"
)

fig.show()